# Import des bibliothèques nécessaires

In [1]:
import pandas as pd
import os
import sys
import pathlib
from pathlib import Path
import spacy

In [2]:
nlp = spacy.load("fr_core_news_lg")

# Chargement des textes dans un DataFrame

In [ ]:
dossier_path = Path("/Users/morganrichard/BDD-Naturaliste-Non-Naturaliste/corpus_goncourt") # Chemin vers le dossier contenant les fichiers .txt

donnees = [] # Liste pour stocker les données extraites de chaque fichier

# On boucle sur tous les fichiers .txt du dossier
for fichier in dossier_path.glob("*.txt"):
    with open(fichier, "r", encoding="utf-8") as f:
        contenu = f.read()
        
        # On ajoute les données extraites du fichier à la liste
    donnees.append({
            "nom_fichier": fichier.name, # Nom du fichier
            "texte_brut": contenu, # Contenu brut du fichier
        })
      
         

# Création du tableau de bord (DataFrame)
df = pd.DataFrame(donnees)
print(f"{len(df)} textes chargés avec succès.")

10 textes chargés avec succès.


In [4]:
df

,nom_fichier,texte_brut
0,1869_madame_gervaisais_travail.txt,"— Quarante scudi ?\n— Oui, signora.\n— Cela fa..."
1,1884_cherie_travail.txt,"petites amies à peu près de son âge, des place..."
2,1861_sœur_philomène_travail.txt,"La salle est haute et vaste. Elle est longue, ..."
3,1879_frères_zemganno_travail.txt,"En pleine campagne, au pied d’un poteau d’octr..."
4,1877_la_fille_elisa_travail.txt,"La femme, la prostituée condamnée à mort, étai..."
5,1865_germinie_lacerteux_travail.txt,"— Sauvée ! vous voilà donc sauvée, mademoisell..."
6,1867_manette_salomon_travail.txt,On était au commencement de novembre. La derni...
7,1860_Charles Demailly_travail.txt,– Un article ?… Tu me demandes s’ y a un artic...
8,1882_la_faustin_travail.txt,"faisait nuit sous un ciel étoilé, au-dessus d'..."
9,1864_renée_mauperin_travail.txt,"— Vous n’aimez pas le monde, mademoiselle ?\n—..."


# Segmentation du texte en segments de 300 mots

In [18]:
def segmenter_texte(text, taille=300): # On segmente le texte en segments de 300 mots
    mots = text.split() # On divise le texte en mots
    segments = [] # Liste pour stocker les segments de texte
    for i in range(0, len(mots), taille): # On boucle sur les mots par pas de "taille" (300 mots)
        segment = " ".join(mots[i:i+taille]) # On crée un segment en joignant les mots du segment
        if len(segment.split()) >= 100:  # On vérifie que le segment contient au moins 100 mots pour éviter les segments trop courts
            segments.append(segment) # On ajoute le segment à la liste des segments
    return segments # On retourne la liste des segments de texte

lignes = []

df_titre = df[df["nom_fichier"] == "1867_manette_salomon_travail.txt"] # On filtre le DataFrame pour ne garder que le texte du fichier que l'on veut segmenter

for _, row in df_titre.iterrows():
    titre = row["nom_fichier"]
    texte = row["texte_brut"]
    segments = segmenter_texte(texte)

    for j, seg in enumerate(segments):
        lignes.append({
            "titre": titre,
            "segment_id": j,
            "texte_segment": seg
        })

df_segments = pd.DataFrame(lignes)
df_segments

,titre,segment_id,texte_segment
0,1867_manette_salomon_travail.txt,0,On était au commencement de novembre. La derni...
1,1867_manette_salomon_travail.txt,1,"embarrassé dans sa culotte, et qui semblait to..."
2,1867_manette_salomon_travail.txt,2,formaient et développaient comme un front de c...
3,1867_manette_salomon_travail.txt,3,"une pelouse. Au-delà de la cime des sapins, un..."
4,1867_manette_salomon_travail.txt,4,"Milady, voilà ! confiez-moi votre œil… Je n’en..."
...,...,...,...
466,1867_manette_salomon_travail.txt,466,"sorte de douceur frissonnante, avec des reflet..."
467,1867_manette_salomon_travail.txt,467,les cigognes prennent des essors boiteux ; et ...
468,1867_manette_salomon_travail.txt,468,"rond, montre, en ouvrant son immense bouche en..."
469,1867_manette_salomon_travail.txt,469,"fleurs de lis d’ombre. Assis sur un banc, sous..."


# Envoie dans la base de données

In [19]:
import mysql.connector

# connexion MySQL
connexion = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Azertyui&É",
    database="Roman_nlp"
)
cursor = connexion.cursor()

id_roman = 5 # ID du roman auquel les segments appartiennent (à adapter selon la base de données)

requete = """
INSERT INTO segments (id_roman, ordre_segment, texte_segment)
VALUES (%s, %s, %s)
"""

for _, row in df_segments.iterrows(): # On boucle sur chaque segment de texte dans le DataFrame
    ordre_segment = int(row["segment_id"]) + 1 # On ajoute 1 pour que l'ordre commence à 1 au lieu de 0
    texte_segment = row["texte_segment"] # Contenu du segment de texte

    cursor.execute(requete,(id_roman, ordre_segment, texte_segment)) 
    # On exécute la requête d'insertion pour chaque segment de texte

connexion.commit() # On valide les changements dans la base de données
cursor.close() # On ferme le curseur
connexion.close() # On ferme la connexion à la base de données

print("Segments ajoutés à la base.")


Segments ajoutés à la base.
